# 10 - Churn Feature Engineering
Builds point-in-time customer features and labels from Gold.

| Table | Purpose | Key |
|---|---|---|
| `ml.customer_features` | Behavior features as of a snapshot date | `customer_key`, `snapshot_date` (time series) |
| `ml.churn_labels` | `churned = 1` if no purchase in the 270 days after the snapshot | `customer_key`, `snapshot_date` |

- Features only use orders **before** the snapshot (no data leakage)
- `customer_key` = SHA-256 hash of `customer_unique_id` (no raw PII in ML tables)
- Snapshots: **2017-12-01** (training, has labels) and **2018-09-01** (scoring, latest data)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.ml
COMMENT 'Machine learning: feature tables, labels, models and predictions';

ALTER SCHEMA retail_lakehouse.ml SET TAGS ('layer' = 'ml', 'data_classification' = 'derived_features');

In [0]:
C = "retail_lakehouse"
TRAIN_SNAPSHOT = "2017-12-01"
SCORE_SNAPSHOT = "2018-09-01"
LABEL_WINDOW_DAYS = 270

VALID_ORDER = "f.order_status NOT IN ('canceled', 'unavailable')"


def build_features(snapshot_date: str):
    """Customer behavior features using ONLY orders before snapshot_date."""
    return spark.sql(f"""
    WITH orders AS (
      SELECT
        sha2(c.customer_unique_id, 256)        AS customer_key,
        f.order_id,
        MIN(f.order_purchase_timestamp)        AS purchased_at,
        SUM(f.item_total)                      AS order_value,
        SUM(f.freight_value)                   AS order_freight,
        COUNT(*)                               AS items,
        MAX(f.delivery_days)                   AS delivery_days,
        MAX(CAST(f.is_late_delivery AS INT))   AS was_late,
        MAX(c.customer_state)                  AS customer_state
      FROM {C}.gold.fact_orders f
      JOIN {C}.gold.dim_customer c ON f.customer_sk = c.customer_sk
      WHERE f.order_purchase_timestamp < DATE'{snapshot_date}' AND {VALID_ORDER}
      GROUP BY 1, 2
    ),
    categories AS (
      SELECT sha2(c.customer_unique_id, 256)   AS customer_key,
             COUNT(DISTINCT p.product_category_name) AS distinct_categories
      FROM {C}.gold.fact_orders f
      JOIN {C}.gold.dim_customer c ON f.customer_sk = c.customer_sk
      JOIN {C}.gold.dim_product  p ON f.product_sk  = p.product_sk
      WHERE f.order_purchase_timestamp < DATE'{snapshot_date}' AND {VALID_ORDER}
      GROUP BY 1
    ),
    pay AS (
      SELECT order_id,
             AVG(payment_installments)                                AS avg_installments,
             MAX(CASE WHEN payment_type = 'credit_card' THEN 1 ELSE 0 END) AS used_credit_card,
             MAX(CASE WHEN payment_type = 'voucher'     THEN 1 ELSE 0 END) AS used_voucher
      FROM {C}.silver.payments
      GROUP BY order_id
    )
    SELECT
      o.customer_key,
      DATE'{snapshot_date}'                                                      AS snapshot_date,
      CAST(datediff(DATE'{snapshot_date}', MAX(o.purchased_at)) AS DOUBLE)       AS recency_days,
      CAST(datediff(DATE'{snapshot_date}', MIN(o.purchased_at)) AS DOUBLE)       AS tenure_days,
      CAST(COUNT(*) AS DOUBLE)                                                   AS frequency,
      CAST(SUM(o.order_value) AS DOUBLE)                                         AS monetary,
      CAST(AVG(o.order_value) AS DOUBLE)                                         AS avg_order_value,
      CAST(AVG(o.items) AS DOUBLE)                                               AS avg_items_per_order,
      CAST(SUM(o.order_freight) / NULLIF(SUM(o.order_value), 0) AS DOUBLE)      AS freight_share,
      CAST(AVG(o.delivery_days) AS DOUBLE)                                       AS avg_delivery_days,
      CAST(AVG(o.was_late) AS DOUBLE)                                            AS late_delivery_rate,
      CAST(AVG(p.avg_installments) AS DOUBLE)                                    AS avg_installments,
      CAST(AVG(p.used_credit_card) AS DOUBLE)                                    AS credit_card_share,
      CAST(AVG(p.used_voucher) AS DOUBLE)                                        AS voucher_share,
      CAST(MAX(cat.distinct_categories) AS DOUBLE)                               AS distinct_categories,
      MAX(o.customer_state)                                                      AS customer_state
    FROM orders o
    LEFT JOIN pay p          ON o.order_id = p.order_id
    LEFT JOIN categories cat ON o.customer_key = cat.customer_key
    GROUP BY o.customer_key
    """)


def build_labels(snapshot_date: str, window_days: int = LABEL_WINDOW_DAYS):
    """churned = 1 if a customer active before the snapshot did NOT buy again within the window."""
    return spark.sql(f"""
    WITH active AS (
      SELECT DISTINCT sha2(c.customer_unique_id, 256) AS customer_key
      FROM {C}.gold.fact_orders f
      JOIN {C}.gold.dim_customer c ON f.customer_sk = c.customer_sk
      WHERE f.order_purchase_timestamp < DATE'{snapshot_date}' AND {VALID_ORDER}
    ),
    returned AS (
      SELECT DISTINCT sha2(c.customer_unique_id, 256) AS customer_key
      FROM {C}.gold.fact_orders f
      JOIN {C}.gold.dim_customer c ON f.customer_sk = c.customer_sk
      WHERE f.order_purchase_timestamp >= DATE'{snapshot_date}'
        AND f.order_purchase_timestamp <  date_add(DATE'{snapshot_date}', {window_days})
        AND {VALID_ORDER}
    )
    SELECT a.customer_key,
           DATE'{snapshot_date}' AS snapshot_date,
           CASE WHEN r.customer_key IS NULL THEN 1 ELSE 0 END AS churned
    FROM active a
    LEFT JOIN returned r ON a.customer_key = r.customer_key
    """)

In [0]:
features = build_features(TRAIN_SNAPSHOT).unionByName(build_features(SCORE_SNAPSHOT))
features.createOrReplaceTempView("features_v")

spark.sql(f"DROP TABLE IF EXISTS {C}.ml.customer_features")
spark.sql(f"""
  CREATE TABLE {C}.ml.customer_features
  COMMENT 'Point-in-time customer behavior features for churn modeling. One row per customer per snapshot.'
  AS SELECT * FROM features_v
""")

# A Unity Catalog table with a primary key is a feature table
spark.sql(f"ALTER TABLE {C}.ml.customer_features ALTER COLUMN customer_key  SET NOT NULL")
spark.sql(f"ALTER TABLE {C}.ml.customer_features ALTER COLUMN snapshot_date SET NOT NULL")
spark.sql(f"""
  ALTER TABLE {C}.ml.customer_features
  ADD CONSTRAINT customer_features_pk PRIMARY KEY (customer_key, snapshot_date TIMESERIES)
""")
spark.sql(f"ALTER TABLE {C}.ml.customer_features SET TAGS ('table_type' = 'feature_table', 'domain' = 'customer')")

display(spark.sql(f"""
  SELECT snapshot_date, COUNT(*) AS customers,
         ROUND(AVG(frequency), 3) AS avg_orders,
         ROUND(AVG(monetary), 2) AS avg_spend,
         ROUND(AVG(recency_days), 0) AS avg_recency_days
  FROM {C}.ml.customer_features
  GROUP BY snapshot_date ORDER BY snapshot_date
"""))

In [0]:
labels = build_labels(TRAIN_SNAPSHOT)
labels.createOrReplaceTempView("labels_v")

spark.sql(f"DROP TABLE IF EXISTS {C}.ml.churn_labels")
spark.sql(f"""
  CREATE TABLE {C}.ml.churn_labels
  COMMENT 'Churn labels: 1 = no purchase in the {LABEL_WINDOW_DAYS} days after snapshot_date.'
  AS SELECT * FROM labels_v
""")

display(spark.sql(f"""
  SELECT snapshot_date,
         COUNT(*)                         AS customers,
         SUM(churned)                     AS churned,
         COUNT(*) - SUM(churned)          AS returned,
         ROUND(AVG(churned) * 100, 2)     AS churn_rate_pct
  FROM {C}.ml.churn_labels
  GROUP BY snapshot_date
"""))